# 🔍 Phase 1 — Exploration du Dataset MVTec 3D-AD
**Projet : Détection d'Anomalies dans des Objets Industriels 3D**

---
**Objectifs de cette phase :**
- Monter Google Drive et extraire le dataset
- Comprendre la structure du dataset MVTec 3D-AD
- Explorer les données 3D (nuages de points `.tiff` + images RGB)
- Visualiser des objets normaux et défectueux
- Produire des statistiques descriptives pour orienter les choix architecturaux

**Dataset :** MVTec 3D-AD — [Référence papier](https://arxiv.org/abs/2112.09045)

## 1. Installation des dépendances

In [ ]:
# Bibliothèques spécialisées en traitement 3D
!pip install open3d tifffile matplotlib numpy pandas plotly -q
print('✅ Dépendances installées')

## 2. Montage de Google Drive et extraction du dataset

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive monté')

In [ ]:
import os

# ─── À ADAPTER selon votre dossier Drive ───────────────────────────────────
DRIVE_WORK_DIR = '/content/drive/MyDrive/Detection_Anomalie_3D'
EXTRACT_DIR    = '/content/mvtec3d'          # destination locale sur Colab
# ────────────────────────────────────────────────────────────────────────────

# Lister les fichiers disponibles dans le dossier de travail
print('📂 Contenu du dossier Drive :')
for f in sorted(os.listdir(DRIVE_WORK_DIR)):
    size = os.path.getsize(os.path.join(DRIVE_WORK_DIR, f))
    print(f'  {f}  ({size/1e9:.2f} GB)' if size > 1e9 else f'  {f}  ({size/1e6:.1f} MB)')

In [ ]:
import glob

# Trouver automatiquement le(s) fichier(s) .tar.xz dans le dossier Drive
archives = glob.glob(os.path.join(DRIVE_WORK_DIR, '*.tar.xz'))
print(f'📦 Archives trouvées : {archives}')

os.makedirs(EXTRACT_DIR, exist_ok=True)

# Extraction (peut prendre quelques minutes selon la taille)
for archive in archives:
    name = os.path.basename(archive)
    print(f'⏳ Extraction de {name} ...')
    !tar -xf "{archive}" -C "{EXTRACT_DIR}"
    print(f'✅ {name} extrait')

print('\n📂 Contenu extrait dans', EXTRACT_DIR)
!ls "{EXTRACT_DIR}"

## 3. Exploration de la structure du dataset

In [ ]:
import os
import pandas as pd
from collections import defaultdict

# ─── Détecter les catégories disponibles ──────────────────────────────────
CATEGORIES = sorted([
    d for d in os.listdir(EXTRACT_DIR)
    if os.path.isdir(os.path.join(EXTRACT_DIR, d))
])
print(f'📦 Catégories trouvées ({len(CATEGORIES)}) :', CATEGORIES)

In [ ]:
def explore_category(base_dir, category):
    """Explore la structure d'une catégorie MVTec 3D-AD et retourne un résumé."""
    cat_path = os.path.join(base_dir, category)
    stats = {'category': category, 'splits': {}}

    for split in ['train', 'validation', 'test']:
        split_path = os.path.join(cat_path, split)
        if not os.path.exists(split_path):
            continue

        defect_types = sorted(os.listdir(split_path))
        split_info = {}

        for defect in defect_types:
            defect_path = os.path.join(split_path, defect)
            if not os.path.isdir(defect_path):
                continue

            # Compter les fichiers par type
            files = os.listdir(defect_path)
            subfolders = [f for f in files if os.path.isdir(os.path.join(defect_path, f))]

            if subfolders:
                # Structure : split/defect/rgb, xyz, gt
                counts = {}
                for sub in subfolders:
                    sub_files = os.listdir(os.path.join(defect_path, sub))
                    counts[sub] = len(sub_files)
                split_info[defect] = counts
            else:
                split_info[defect] = {'files': len(files)}

        stats['splits'][split] = split_info
    return stats


# Explorer toutes les catégories
all_stats = [explore_category(EXTRACT_DIR, cat) for cat in CATEGORIES]
print('✅ Structure explorée pour', len(all_stats), 'catégorie(s)')

In [ ]:
# Afficher un résumé lisible par catégorie
rows = []
for stat in all_stats:
    cat = stat['category']
    for split, defects in stat['splits'].items():
        for defect, counts in defects.items():
            n = counts.get('rgb', counts.get('xyz', counts.get('files', 0)))
            rows.append({'Catégorie': cat, 'Split': split, 'Type défaut': defect, 'N échantillons': n})

df = pd.DataFrame(rows)
print('=== Résumé global du dataset ===')
print(df.groupby(['Catégorie', 'Split'])['N échantillons'].sum().unstack(fill_value=0).to_string())

In [ ]:
# Afficher les types de défauts par catégorie
print('\n=== Types de défauts par catégorie ===')
for stat in all_stats:
    cat = stat['category']
    test_defects = list(stat['splits'].get('test', {}).keys())
    print(f'  {cat:15s} → {test_defects}')

## 4. Sélection de la catégorie de travail

In [ ]:
# ─── Choisir la catégorie après exploration ────────────────────────────────
# Modifiez cette variable après avoir vu le résumé ci-dessus
CATEGORY = CATEGORIES[0]   # ex: 'bagel', 'cable_gland', 'foam' ...
# ────────────────────────────────────────────────────────────────────────────

TRAIN_GOOD_DIR = os.path.join(EXTRACT_DIR, CATEGORY, 'train', 'good')
TEST_DIR       = os.path.join(EXTRACT_DIR, CATEGORY, 'test')

print(f'📌 Catégorie sélectionnée : {CATEGORY}')
print(f'   Train/good : {TRAIN_GOOD_DIR}')
print(f'   Test       : {TEST_DIR}')

# Lister la structure interne
print('\n🗂  Structure interne :')
for root, dirs, files in os.walk(os.path.join(EXTRACT_DIR, CATEGORY)):
    depth = root.replace(os.path.join(EXTRACT_DIR, CATEGORY), '').count(os.sep)
    if depth > 2:
        continue
    indent = '  ' * depth
    folder = os.path.basename(root)
    n = len([f for f in files])
    print(f'{indent}📁 {folder}/ ({n} fichiers)' if n else f'{indent}📁 {folder}/')

## 5. Chargement et visualisation des données RGB

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tifffile
from pathlib import Path

def get_sample_paths(split_dir, subfolder='rgb', ext='.png', n=4):
    """Récupère n chemins de fichiers dans split_dir/*/subfolder/."""
    paths = []
    split_path = Path(split_dir)
    for defect_dir in sorted(split_path.iterdir()):
        if not defect_dir.is_dir():
            continue
        sub = defect_dir / subfolder
        if sub.exists():
            files = sorted(sub.glob(f'*{ext}'))
            paths.extend([(str(f), defect_dir.name) for f in files[:n]])
    return paths


def show_rgb_grid(paths, title='Exemples RGB', cols=4):
    """Affiche une grille d'images RGB avec leurs labels."""
    n = len(paths)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))
    axes = axes.flatten() if n > 1 else [axes]

    for ax, (path, label) in zip(axes, paths):
        img = plt.imread(path)
        ax.imshow(img)
        color = 'green' if label == 'good' else 'red'
        ax.set_title(label, color=color, fontsize=9)
        ax.axis('off')

    for ax in axes[n:]:
        ax.axis('off')

    fig.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()


# Afficher des images de train (good) et de test (défauts)
train_rgb = get_sample_paths(TRAIN_GOOD_DIR, subfolder='rgb', n=4)

# Adaptation si train/good contient directement les images (pas de sous-dossier rgb)
if not train_rgb:
    rgb_files = sorted(Path(TRAIN_GOOD_DIR).glob('rgb/*.png'))[:4]
    train_rgb = [(str(f), 'good') for f in rgb_files]

print(f'✅ {len(train_rgb)} images train/good trouvées')
if train_rgb:
    show_rgb_grid(train_rgb, title=f'{CATEGORY} — Pièces normales (train)')

In [ ]:
# Images de test (toutes classes de défauts)
test_rgb = []
for defect_dir in sorted(Path(TEST_DIR).iterdir()):
    rgb_dir = defect_dir / 'rgb'
    if rgb_dir.exists():
        files = sorted(rgb_dir.glob('*.png'))[:2]
        test_rgb.extend([(str(f), defect_dir.name) for f in files])

print(f'✅ {len(test_rgb)} images de test trouvées')
if test_rgb:
    show_rgb_grid(test_rgb, title=f'{CATEGORY} — Pièces de test (normal + défauts)', cols=4)

## 6. Chargement et visualisation des nuages de points 3D (XYZ)

In [ ]:
import tifffile
import open3d as o3d

def load_xyz_tiff(path):
    """
    Charge un fichier .tiff contenant un nuage de points 3D (format MVTec 3D-AD).
    Chaque pixel (H, W) contient (X, Y, Z) — les pixels invalides sont à 0.
    Retourne un tableau numpy (N, 3) de points valides.
    """
    xyz_map = tifffile.imread(path)          # shape : (H, W, 3)
    mask = np.any(xyz_map != 0, axis=-1)     # ignorer les pixels nuls
    points = xyz_map[mask]                   # (N, 3)
    return points, xyz_map


def find_xyz_files(base_path, defect_name='good', n=1):
    """Cherche les fichiers .tiff xyz dans la structure MVTec."""
    base = Path(base_path)
    candidates = []

    # Structure : base/xyz/*.tiff
    xyz_dir = base / 'xyz'
    if xyz_dir.exists():
        candidates = sorted(xyz_dir.glob('*.tiff'))[:n]

    # Structure : base/defect_name/xyz/*.tiff
    if not candidates:
        xyz_dir = base / defect_name / 'xyz'
        if xyz_dir.exists():
            candidates = sorted(xyz_dir.glob('*.tiff'))[:n]

    return [str(p) for p in candidates]


# Charger un exemple good du train
xyz_paths_good = find_xyz_files(TRAIN_GOOD_DIR, n=1)

# Fallback : chercher directement dans train/good/xyz
if not xyz_paths_good:
    p = Path(TRAIN_GOOD_DIR)
    xyz_paths_good = [str(f) for f in sorted(p.rglob('*.tiff'))[:1]]

print(f'📄 Fichier XYZ trouvé : {xyz_paths_good}')

if xyz_paths_good:
    pts_good, xyz_map_good = load_xyz_tiff(xyz_paths_good[0])
    print(f'  → Shape de la carte XYZ : {xyz_map_good.shape}')
    print(f'  → Points 3D valides : {pts_good.shape[0]:,}')
    print(f'  → Plage X : [{pts_good[:,0].min():.3f}, {pts_good[:,0].max():.3f}]')
    print(f'  → Plage Y : [{pts_good[:,1].min():.3f}, {pts_good[:,1].max():.3f}]')
    print(f'  → Plage Z : [{pts_good[:,2].min():.3f}, {pts_good[:,2].max():.3f}]')

In [ ]:
import plotly.graph_objects as go

def visualize_pointcloud_plotly(points, title='Nuage de points 3D', max_pts=5000, color_by='z'):
    """
    Visualisation interactive du nuage de points avec Plotly.
    Sous-échantillonne si nécessaire pour la fluidité.
    """
    if len(points) > max_pts:
        idx = np.random.choice(len(points), max_pts, replace=False)
        pts = points[idx]
    else:
        pts = points

    # Couleur selon l'axe Z (profondeur)
    color = pts[:, 2] if color_by == 'z' else pts[:, 0]

    fig = go.Figure(data=[
        go.Scatter3d(
            x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
            mode='markers',
            marker=dict(size=1.5, color=color, colorscale='Viridis',
                        colorbar=dict(title='Z'), opacity=0.8)
        )
    ])
    fig.update_layout(
        title=title,
        scene=dict(xaxis_title='X', yaxis_title='Y', zaxis_title='Z'),
        width=750, height=550
    )
    fig.show()


if xyz_paths_good:
    visualize_pointcloud_plotly(
        pts_good,
        title=f'{CATEGORY} — Pièce normale (nuage de points)',
        max_pts=8000
    )

In [ ]:
# Comparaison good vs défectueux côte à côte (vue Z projection)
def find_test_xyz(test_dir, n_per_class=1):
    """Retourne un fichier xyz par type de défaut dans le test."""
    results = []
    for defect_dir in sorted(Path(test_dir).iterdir()):
        xyz_dir = defect_dir / 'xyz'
        if xyz_dir.exists():
            files = sorted(xyz_dir.glob('*.tiff'))[:n_per_class]
            results.extend([(str(f), defect_dir.name) for f in files])
    return results


test_xyz_paths = find_test_xyz(TEST_DIR, n_per_class=1)
print(f'✅ {len(test_xyz_paths)} fichiers XYZ de test trouvés')

# Visualisation de la projection Z (vue de dessus) pour comparaison rapide
n_samples = min(len(test_xyz_paths), 6)
if n_samples > 0:
    fig, axes = plt.subplots(1, n_samples, figsize=(4 * n_samples, 4))
    if n_samples == 1:
        axes = [axes]

    for ax, (path, label) in zip(axes, test_xyz_paths[:n_samples]):
        _, xyz_map = load_xyz_tiff(path)
        z_channel = xyz_map[:, :, 2]
        z_channel[z_channel == 0] = np.nan
        ax.imshow(z_channel, cmap='plasma')
        color = 'green' if label == 'good' else 'red'
        ax.set_title(label, color=color, fontsize=9)
        ax.axis('off')

    plt.suptitle(f'{CATEGORY} — Projection Z (profondeur) des pièces de test', fontsize=12)
    plt.tight_layout()
    plt.show()

## 7. Visualisation des masques de vérité terrain (Ground Truth)

In [ ]:
def show_gt_comparison(test_dir, n=3):
    """
    Affiche côte à côte : image RGB | projection Z | masque GT
    pour les pièces défectueuses du test.
    """
    samples = []
    for defect_dir in sorted(Path(test_dir).iterdir()):
        if defect_dir.name == 'good':
            continue
        rgb_dir = defect_dir / 'rgb'
        xyz_dir = defect_dir / 'xyz'
        gt_dir  = defect_dir / 'gt'

        if rgb_dir.exists() and xyz_dir.exists() and gt_dir.exists():
            rgbs = sorted(rgb_dir.glob('*.png'))
            xyzs = sorted(xyz_dir.glob('*.tiff'))
            gts  = sorted(gt_dir.glob('*.png'))
            for r, x, g in zip(rgbs, xyzs, gts):
                samples.append((str(r), str(x), str(g), defect_dir.name))
                if len(samples) >= n:
                    break
        if len(samples) >= n:
            break

    if not samples:
        print('⚠️  Aucun échantillon avec GT trouvé. Vérifiez la structure du dossier test.')
        return

    fig, axes = plt.subplots(len(samples), 3, figsize=(10, 3.5 * len(samples)))
    if len(samples) == 1:
        axes = [axes]

    for row, (rgb_p, xyz_p, gt_p, label) in enumerate(samples):
        rgb_img = plt.imread(rgb_p)
        _, xyz_map = load_xyz_tiff(xyz_p)
        z_ch = xyz_map[:, :, 2]
        z_ch[z_ch == 0] = np.nan
        gt_img = plt.imread(gt_p)

        axes[row][0].imshow(rgb_img)
        axes[row][0].set_title(f'RGB — {label}', fontsize=9)
        axes[row][0].axis('off')

        axes[row][1].imshow(z_ch, cmap='plasma')
        axes[row][1].set_title('Profondeur Z', fontsize=9)
        axes[row][1].axis('off')

        axes[row][2].imshow(gt_img, cmap='Reds')
        axes[row][2].set_title('Masque GT (anomalie)', fontsize=9)
        axes[row][2].axis('off')

    plt.suptitle(f'{CATEGORY} — RGB | Profondeur | Masque de vérité terrain', fontsize=12)
    plt.tight_layout()
    plt.show()


show_gt_comparison(TEST_DIR, n=3)

## 8. Statistiques descriptives du dataset

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ── Comptage des échantillons par split et type de défaut ──────────────────
split_counts = {'train': {}, 'validation': {}, 'test': {}}

cat_path = Path(EXTRACT_DIR) / CATEGORY
for split in split_counts:
    split_path = cat_path / split
    if not split_path.exists():
        continue
    for defect_dir in sorted(split_path.iterdir()):
        xyz_dir = defect_dir / 'xyz'
        rgb_dir = defect_dir / 'rgb'
        ref_dir = xyz_dir if xyz_dir.exists() else rgb_dir
        if ref_dir.exists():
            n = len(list(ref_dir.glob('*.*')))
            split_counts[split][defect_dir.name] = n

# Affichage tableau
print(f'\n=== Statistiques — {CATEGORY} ===\n')
for split, counts in split_counts.items():
    if counts:
        total = sum(counts.values())
        print(f'  [{split}]  total={total}')
        for k, v in counts.items():
            tag = '✓' if k == 'good' else '✗'
            print(f'    {tag} {k:20s} : {v:4d}')
    else:
        print(f'  [{split}]  (vide ou absent)')

In [ ]:
# ── Visualisation de la distribution ──────────────────────────────────────
test_counts = split_counts.get('test', {})
if test_counts:
    labels = list(test_counts.keys())
    values = list(test_counts.values())
    colors = ['#2ecc71' if l == 'good' else '#e74c3c' for l in labels]

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Barplot
    axes[0].bar(labels, values, color=colors, edgecolor='white', linewidth=0.5)
    axes[0].set_title(f'{CATEGORY} — Distribution test par type de défaut')
    axes[0].set_xlabel('Type de défaut')
    axes[0].set_ylabel('Nombre d\'échantillons')
    axes[0].tick_params(axis='x', rotation=30)

    # Camembert good vs défectueux
    n_good   = test_counts.get('good', 0)
    n_defect = sum(v for k, v in test_counts.items() if k != 'good')
    axes[1].pie([n_good, n_defect],
                labels=['Normal', 'Défectueux'],
                colors=['#2ecc71', '#e74c3c'],
                autopct='%1.1f%%', startangle=90)
    axes[1].set_title(f'{CATEGORY} — Équilibre des classes (test)')

    plt.tight_layout()
    plt.show()

    print(f'\n⚖️  Déséquilibre : {n_good} normaux / {n_defect} défectueux')
    if n_defect > 0:
        ratio = n_good / n_defect
        print(f'   Ratio normal/défaut : {ratio:.2f}')
        if ratio > 2:
            print('   ⚠️  Déséquilibre significatif → à prendre en compte dans la modélisation')

In [ ]:
# ── Distribution des valeurs Z (profondeur) sur un échantillon ─────────────
if xyz_paths_good:
    pts, _ = load_xyz_tiff(xyz_paths_good[0])

    fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
    for i, (ax, axis_name) in enumerate(zip(axes, ['X', 'Y', 'Z'])):
        ax.hist(pts[:, i], bins=80, color=['#3498db', '#2ecc71', '#e74c3c'][i],
                alpha=0.8, edgecolor='white', linewidth=0.3)
        ax.set_title(f'Distribution axe {axis_name}')
        ax.set_xlabel(f'{axis_name} (mm ou unité brute)')
        ax.set_ylabel('Fréquence')

    plt.suptitle(f'{CATEGORY} — Distribution des coordonnées 3D (pièce normale)', fontsize=11)
    plt.tight_layout()
    plt.show()

    print(f'\n📊 Statistiques des points 3D (pièce normale) :')
    for i, ax_name in enumerate(['X', 'Y', 'Z']):
        col = pts[:, i]
        print(f'  {ax_name} — min={col.min():.4f}  max={col.max():.4f}  '
              f'mean={col.mean():.4f}  std={col.std():.4f}')

## 9. Bilan & Décisions pour la Phase 2

In [ ]:
# Récapitulatif automatique pour orienter la suite
n_train = sum(split_counts.get('train', {}).values())
n_val   = sum(split_counts.get('validation', {}).values())
n_test  = sum(split_counts.get('test', {}).values())
defect_types = [k for k in split_counts.get('test', {}) if k != 'good']

print('=' * 55)
print(f'  BILAN PHASE 1 — Catégorie : {CATEGORY}')
print('=' * 55)
print(f'  Train         : {n_train} échantillons (normal uniquement)')
print(f'  Validation    : {n_val} échantillons')
print(f'  Test          : {n_test} échantillons')
print(f'  Types défauts : {defect_types}')
print()
print('  → Format 3D  : nuages de points via TIFF (H×W×3)')
print('  → Format 2D  : images RGB .png associées')
print('  → GT          : masques binaires .png')
print()
print('  PROCHAINES ÉTAPES (Phase 2) :')
print('  1. Normalisation et recentrage des nuages de points')
print('  2. Sous-échantillonnage uniforme (FPS ou random)')
print('  3. Augmentation données (rotation, bruit, jitter)')
print('  4. Construction des DataLoaders PyTorch')
print('=' * 55)